# Task 1 · Find Where the Image Changes (20 marks)

ES666 Assignment 3 · Lunar Landing: The Final Descent. The full assignment is
in the handout, `ES666_A3_Handout.pdf`, and in `tasks.md`.

Learn how motion is written into image intensities: compute image gradients,
connect them to motion, and find the points that can be tracked. Images are
arrays indexed `image[y, x]` (y is the row, x the column); points are `[x, y]`.
Do not use `np.gradient`.

**How this notebook works**

* Write your code only in the cells whose first line starts with `# YOUR CODE`.
  Only those cells are graded, and they are read from the **saved** notebook.
  Add your own cells anywhere for experiments or notes: they are never graded.
  A helper function you want graded with your code goes in a `# YOUR CODE` cell
  (a new cell whose first line is `# YOUR CODE` works too).
* Run the `check(...)` cell after your code: it tests your functions and prints
  PASS or FAIL for each test.
* Answer each question in the cell right below it, in one or two sentences,
  replacing *TODO*.
* Save often (Ctrl+S). Before you submit, run everything from a clean start:
  restart the kernel and run all cells, or run `python run_notebooks.py`.

**Name:** *TODO*

**Roll number:** *TODO*

In [ ]:
# PROVIDED: run this cell first
%matplotlib inline
import numpy as np
from provided.notebook import check
from provided.task1.steps import check_constraint, load_task_frame, show_features, show_gradients

frame = load_task_frame()          # frame 276 of the descent, at quarter size

## 1.1 Spatial gradients (6 marks)

Implement central differences, without `np.gradient`:

$$
I_x(x,y)=\frac{I(x+1,y)-I(x-1,y)}{2},\qquad I_y(x,y)=\frac{I(x,y+1)-I(x,y-1)}{2}.
$$

In [ ]:
# YOUR CODE: Task 1.1

def spatial_gradients(image):
    """Central differences, equation (1.1):
        Ix(x, y) = (I(x+1, y) - I(x-1, y)) / 2,   Iy(x, y) = (I(x, y+1) - I(x, y-1)) / 2

    Returns (ix, iy): float arrays shaped like image, 0 on the one-pixel border.

    How to write it (no loops needed):
      1. image = image.astype(np.float64); start from ix = np.zeros_like(image).
      2. image[1:-1, 2:] - image[1:-1, :-2] is I(x+1) - I(x-1) for every inner
         pixel: halve it and store it in ix[1:-1, 1:-1], so the whole border stays 0.
      3. Do the same with the neighbours above and below,
         image[2:, 1:-1] - image[:-2, 1:-1], for iy[1:-1, 1:-1].
    """
    # TODO
    raise NotImplementedError("Student implementation required.")

In [ ]:
check("1.1")

In [ ]:
grad = show_gradients(frame, spatial_gradients)

**(a)** Where are the gradients strongest, and where are they weakest?

*TODO*

## 1.2 Temporal change and motion (5 marks)

Implement $I_t = I_1 - I_0$. For a small motion $(u, v)$, brightness constancy
gives the motion constraint

$$
I_x u + I_y v + I_t \approx 0.
$$

The provided cell checks it on frame 276 shifted by a known $(u, v) = (0.3, -0.2)$.

In [ ]:
# YOUR CODE: Task 1.2

def temporal_difference(image0, image1):
    """It = I1 - I0 (Task 1.2): how much each pixel's brightness changed.

    Returns a float array shaped like image0.

    Hint: convert both images with .astype(np.float64) before subtracting, so
    integer images cannot wrap around.
    """
    # TODO
    raise NotImplementedError("Student implementation required.")

In [ ]:
check("1.2")

In [ ]:
stats = check_constraint(frame, spatial_gradients, temporal_difference)

**(a)** What mean residual does the check report (saved in `constraint_check.json`)?

*TODO*

**(b)** Why can one pixel not give both $u$ and $v$?

*TODO*

## 1.3 Choosing good points to track (9 marks)

For each local window build $G = A^TA$, with one row $[I_x, I_y]$ of $A$ per
pixel, and use the smaller eigenvalue as a trackability score:

$$
G = A^TA, \qquad q(x,y)=\min(\lambda_1,\lambda_2).
$$

Select strong points with a minimum separation. The same matrix $G$ is what
Lucas-Kanade solves in Task 2.

In [ ]:
# YOUR CODE: Task 1.3

def structure_tensor(ix, iy, x, y, window_radius):
    """G = A^T A for the window centred on pixel (x, y), equation (1.3).

    The window is rows y-r..y+r and columns x-r..x+r, with r = window_radius.
    Returns the 2x2 array [[sum Ix*Ix, sum Ix*Iy], [sum Ix*Iy, sum Iy*Iy]].

    How to write it:
      1. Cut the windows: wx = ix[y - r:y + r + 1, x - r:x + r + 1], and wy from iy.
      2. Sum the element-wise products (np.sum(wx * wx), ...) into a 2x2 np.array.
    """
    # TODO
    raise NotImplementedError("Student implementation required.")


def feature_score(G):
    """Trackability q = min(lambda1, lambda2) of the 2x2 matrix G, equation (1.3).

    Hint: np.linalg.eigvalsh(G) returns the eigenvalues of a symmetric matrix in
    ascending order, so q is its first element. Return it as a float.
    """
    # TODO
    raise NotImplementedError("Student implementation required.")


def select_features(image, max_points=200, window_radius=3, min_distance=10, threshold=None):
    """Choose up to max_points strong, well-separated points in an image.

    Returns an (N, 2) float array of [x, y], strongest first, N <= max_points.

    How to write it:
      1. ix, iy = spatial_gradients(image).
      2. Score every pixel whose window fits inside the image (r <= x < W - r and
         r <= y < H - r, r = window_radius) with
         feature_score(structure_tensor(ix, iy, x, y, r)).
      3. Keep scores >= threshold if one is given; otherwise drop scores of 0
         (flat areas).
      4. Sort strongest first: order = np.argsort(-scores).
      5. Walk through that order and keep a point only if it is at least
         min_distance from every point kept so far; stop at max_points.

    Speed: Task 4 calls this about 570 times (once per frame pair and tracker).
    A Python loop over every pixel takes about 5 s per frame, which adds about
    45 minutes to Task 4. Start with the loop if you like, then, before Task 4,
    score all windows at once: sum Ix*Ix, Ix*Iy and Iy*Iy over every window with
    an integral image (np.cumsum along both axes), and use the closed form of the
    smaller eigenvalue of [[a, b], [b, c]]: (a + c) / 2 - sqrt(((a - c) / 2)**2 + b**2).
    """
    # TODO
    raise NotImplementedError("Student implementation required.")

In [ ]:
check("1.3")

In [ ]:
points = show_features(frame, select_features)

**(a)** What does the minimum distance between selected points prevent?

*TODO*

**(b)** What do the eigenvalues of $G$ look like on a flat patch, an edge and a corner?

*TODO*

**(c)** Why does a large $q$ mean a point is easy to track?

*TODO*